# PegInsertionSide-v1 — Interactive Notebook

Play with scripted primitives, record rollouts, and debug peg alignment.

This notebook attempts to create the ManiSkill2 `PegInsertionSide-v1` environment, run a simple scripted policy, and record a video inline. Adapt keys and gains to your local installation and observations.

In [221]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import io
import base64
import numpy as np
import imageio
import gymnasium as gym
import torch
import sapien
from mani_skill.utils.structs.pose import Pose
from IPython.display import HTML, display
from mani_skill.utils.geometry import rotation_conversions as RC

def display_video(path, width=640):
    mp4 = open(path,'rb').read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
    html = f'<video width="{width}" controls><source src="{data_url}" type="video/mp4"></video>'
    display(HTML(html))

def frames_to_mp4(frames, path, fps=30):
    # print(len(frames))
    # frames = frames.numpy()
    # Write frames (H x W x C uint8) to an mp4 file using imageio
    print(type(frames), len(frames), frames[0].shape if frames else 'No frames')
    imageio.mimwrite(path, frames, fps=fps, macro_block_size=None)


In [222]:
def capture_frame_from_env(env):
    "Try several common render calls to get an RGB frame (H,W,3)"
    try:
        # common gym-style render
        frame = env.render()
        if frame is None:
            frame = None
    except Exception as e:
        print(f"gym render failed: {e}")
        frame = None
    if frame is None:
        try:
            # ManiSkill / MuJoCo direct render fallback (may vary by build)
            frame = env.sim.render(width=640, height=480, camera_name='agentview')
        except Exception as e:
            print(f"ManiSkill render failed: {e}")
            frame = None
    return frame[0]

In [223]:
global_frames = []
class InsertionCurriculumWrapper(gym.Wrapper):
    """
    Intercepts env.reset() to execute a scripted batched policy that grasps 
    the peg from the table and aligns it with the hole before handing 
    control to the RL agent.
    """
    def __init__(self, env, setup_steps=90):
        super().__init__(env)
        self.setup_steps = setup_steps

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        global_frames.append(capture_frame_from_env(self.env).cpu().numpy())
        device = obs.device if isinstance(obs, torch.Tensor) else "cpu"
        n_envs = getattr(self.env.unwrapped, "num_envs", 1)
        action_dim = self.env.action_space.shape[-1]
        
        for step in range(self.setup_steps):
            # 1. Get current gripper position
            tcp_pos = self.env.unwrapped.agent.tcp.pose.p
            
            # 2. Get target grasp pose from the environment's own reward logic
            # The env explicitly uses an offset of [-0.06, 0, 0] to grab the peg tail
            tgt_gripper_pos = (self.env.unwrapped.peg.pose * sapien.Pose([-0.06, 0, 0])).p

            
            # 3. Get pre-insertion pose
            # The hole points along the local +X axis of box_hole_pose. 
            # We hover 15cm (-0.15) outside the hole.
            pre_insert_pos = (self.env.unwrapped.box_hole_pose * sapien.Pose([-0.15, 0, 0])).p
            
            print('tgt_gripper_pos:', tgt_gripper_pos, 'pre_insert_pos:', pre_insert_pos)
            # 4. Phase-based state machine
            if step < 20:
                # Phase A: Hover 10cm directly above the peg
                target_pos = tgt_gripper_pos + torch.tensor([0, 0, 0.1], device=device)
                gripper_act = 1.0 # Open
            elif step < 40:
                # Phase B: Drop down to the peg tail
                target_pos = tgt_gripper_pos
                gripper_act = 1.0 # Open
            elif step < 60:
                # Phase C: Close the gripper firmly
                target_pos = tgt_gripper_pos
                gripper_act = -1.0 # Close
            else:
                # Phase D: Fly to the hole
                target_pos = pre_insert_pos
                gripper_act = -1.0 # Keep closed
                
            # Proportional (P) controller for smooth movement
            delta_pos = (target_pos - tcp_pos) * 5.0
            delta_pos = torch.clamp(delta_pos, -1.0, 1.0)
            
            # Construct the native action
            scripted_action = torch.zeros((n_envs, action_dim), device=device)
            scripted_action[:, :3] = delta_pos
            scripted_action[:, -1] = gripper_act
            
            # Step the underlying environment silently
            obs, _, _, _, info = self.env.step(scripted_action)
            global_frames.append(capture_frame_from_env(self.env).cpu().numpy())

        # Control is handed to the RL agent! The robot is holding the peg right in front of the hole.
        return obs, info
class ManiskillTeleportWrapper(gym.Wrapper):
    """
    Bypasses the P-controller entirely! Instantly teleports the peg into the robot's 
    hand, and teleports the box directly in front of the robot. 
    """
    def __init__(self, env, setup_steps=15):
        super().__init__(env)
        self.setup_steps = setup_steps

    def reset(self, seed=None, options=None):
        # 1. Reset the environment normally (randomizes shapes and sizes)
        obs, info = self.env.reset(seed=seed, options=options)
        global_frames.append(capture_frame_from_env(self.env).cpu().numpy())
        
        env_unwrapped = self.env.unwrapped
        device = obs.device if isinstance(obs, torch.Tensor) else "cpu"
        n_envs = getattr(env_unwrapped, "num_envs", 1)
        action_dim = self.env.action_space.shape[-1]
        
        # 2. Teleport the Peg into the gripper
        tcp_pose = env_unwrapped.agent.tcp.pose
        
        # The env's reward logic assumes a grasp offset of [-0.06, 0, 0] relative to the peg.
        # By inverting this, we put the peg perfectly inside the TCP.
        offset_p = torch.zeros((n_envs, 3), device=device)
        offset_p[:, 0] = 0.05
        peg_offset = Pose.create_from_pq(p=offset_p)
        
        new_peg_pose = tcp_pose * peg_offset
        env_unwrapped.peg.set_pose(new_peg_pose)
        
        # 3. Teleport the Box directly in front of the gripper
        peg_lengths = env_unwrapped.peg_half_sizes[:, 0]
        
        # Place the hole exactly 2cm (0.02) in front of the peg tip
        hole_offset_p = torch.zeros((n_envs, 3), device=device)
        hole_offset_p[:, 0] = 0.06 + peg_lengths + 0.08
        
        hole_target_pose = tcp_pose * Pose.create_from_pq(p=hole_offset_p)
        
        # Apply the inverse hole offset to perfectly position the outer box
        new_box_pose = hole_target_pose * env_unwrapped.box_hole_offsets.inv()
        env_unwrapped.box.set_pose(new_box_pose)
        
        # 4. Settle the physics (Close the gripper tightly)
        scripted_action = torch.zeros((n_envs, action_dim), device=device)
        scripted_action[:, -1] = -1.0 # Force gripper closed
        
        for _ in range(self.setup_steps):
            obs, _, _, _, info = self.env.step(scripted_action)
            global_frames.append(capture_frame_from_env(self.env).cpu().numpy())
    

        # Hand control to the RL Agent!
        return obs, info

import torch
import gymnasium as gym

import torch
import gymnasium as gym
import numpy as np

# --- Pure PyTorch Vectorized Quaternion Math ---
def quat_mul(q1, q2):
    w1, x1, y1, z1 = q1[:, 0], q1[:, 1], q1[:, 2], q1[:, 3]
    w2, x2, y2, z2 = q2[:, 0], q2[:, 1], q2[:, 2], q2[:, 3]
    return torch.stack([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], dim=-1)

def quat_inv(q):
    q_inv = q.clone()
    q_inv[:, 1:] = -q_inv[:, 1:]
    return q_inv

def quat_to_axis_angle(q):
    # Enforce shortest path rotation
    q = torch.where(q[:, 0:1] < 0, -q, q)
    q = q / torch.norm(q, dim=-1, keepdim=True)
    angle = 2 * torch.acos(torch.clamp(q[:, 0], -1.0, 1.0))
    sin_half_angle = torch.sqrt(torch.clamp(1.0 - q[:, 0]**2, min=1e-8))
    axis = q[:, 1:] / sin_half_angle.unsqueeze(-1)
    return axis * angle.unsqueeze(-1)


class ManiskillTeleportWrapper(gym.Wrapper):
    """
    1. Teleports the peg perfectly into the gripper (aligned with fingers).
    2. Uses a robust PyTorch P-Controller to fly the arm and twist the wrist.
    3. Records the entire setup phase to a self.frames list if enable_rendering=True.
    """
    def __init__(self, env, setup_steps=120, hover_clearance=0.06, enable_rendering=False):
        super().__init__(env)
        self.setup_steps = setup_steps
        self.hover_clearance = hover_clearance
        self.enable_rendering = enable_rendering
        print(self.enable_rendering)
        self.frames = [] # The global frame buffer for this environment

    def _capture_frame(self):
        """Helper to safely extract the RGB image from ManiSkill's renderer."""
        if not self.enable_rendering:
            return None
            
        try:
            # SAPIEN renderer returns a PyTorch tensor on the GPU
            render_out = self.env.unwrapped.render()
            if render_out is None:
                return None
                
            # If batch size > 1, render() returns a batched tensor. We take the first env [0].
            if isinstance(render_out, torch.Tensor):
                # Ensure it's detached, on CPU, and formatted as a numpy array
                if render_out.ndim == 4: # (B, H, W, C)
                    frame = render_out[0].detach().cpu().numpy()
                else: # (H, W, C)
                    frame = render_out.detach().cpu().numpy()
            elif isinstance(render_out, np.ndarray):
                if render_out.ndim == 4:
                    frame = render_out[0]
                else:
                    frame = render_out
            else:
                return None
                
            # Convert to uint8 if it's currently a float array [0, 1]
            if frame.dtype in [np.float32, np.float64]:
                frame = (frame * 255).astype(np.uint8)
            return frame
        except Exception as e:
            print(f"Frame capture failed: {e}")
            return None

    def reset(self, seed=None, options=None):
        self.frames.clear() # ALWAYS clear before starting a new episode!
        
        obs, info = self.env.reset(seed=seed, options=options)
        
        # Capture the initial raw state
        f = self._capture_frame()
        if f is not None: self.frames.append(f)
        
        env_unwrapped = self.env.unwrapped
        device = obs.device if isinstance(obs, torch.Tensor) else "cpu"
        n_envs = getattr(env_unwrapped, "num_envs", 1)
        action_dim = self.env.action_space.shape[-1]
        PoseClass = type(env_unwrapped.agent.tcp.pose)
        
        # --- PHASE 0: OPEN THE GRIPPER WIDE ---
        open_action = torch.zeros((n_envs, action_dim), device=device)
        open_action[:, -1] = 1.0 
        for _ in range(10):
            self.env.step(open_action)
            f = self._capture_frame()
            if f is not None: self.frames.append(f)

        # --- PHASE 1 & 2: TELEPORT AND SETTLE (WITH ANTI-GRAVITY LOCK) ---
        tcp_pose = env_unwrapped.agent.tcp.pose
        angle = 0
        grasp_q = torch.tensor([[np.cos(angle/2), 0.0, 0.0, np.sin(angle/2)]], device=device).repeat(n_envs, 1)
        grasp_p = torch.zeros((n_envs, 3), device=device)
        grasp_p[:, 0] = 0.05
        # grasp_p[:, 1] = 0.03
        grasp_p[:, 2] = -0.02
        
        peg_offset = PoseClass.create_from_pq(p=grasp_p, q=grasp_q)
        new_peg_pose = tcp_pose * peg_offset
        env_unwrapped.peg.set_pose(new_peg_pose)
        # env_unwrapped.peg.set_linear_velocity(torch.zeros((n_envs, 3), device=device))
        # env_unwrapped.peg.set_angular_velocity(torch.zeros((n_envs, 3), device=device))

        settle_action = torch.zeros((n_envs, action_dim), device=device)
        settle_action[:, -1] = -1.0 
        obs, _, _, _, info = self.env.step(settle_action)

        f = self._capture_frame()
        self.frames.append(f)

        # for i in range(10):
        #     if i == 1: # Teleport the peg on the first step of this phase
        #         print('Teleporting peg to:', new_peg_pose)
                
        #     f = self._capture_frame()
        #     if f is not None: self.frames.append(f)

        # --- PHASE 3: FLY TO HOLE ---
        peg_lengths = env_unwrapped.peg_half_sizes[:, 0]
        print('peg_lengths:', peg_lengths)
        pos_gain = 10.0 * 0.1
        ori_gain = 6.0 * 0.05
        
        for step in range(self.setup_steps):
            tcp_pose = env_unwrapped.agent.tcp.pose
            hole_pose = env_unwrapped.box.pose * env_unwrapped.box_hole_offsets
            hover_offset_p = torch.zeros((n_envs, 3), device=device)
            hover_offset_p[:, 0] = -(0.06 + peg_lengths + self.hover_clearance)
            
            target_peg_pose = hole_pose * PoseClass.create_from_pq(p=hover_offset_p)
            target_tcp_pose = target_peg_pose * peg_offset.inv()
            
            pos_error = target_tcp_pose.p - tcp_pose.p
            
            q_err = quat_mul(target_tcp_pose.q, quat_inv(tcp_pose.q))
            ori_error = quat_to_axis_angle(q_err)
            
            scripted_action = torch.zeros((n_envs, action_dim), device=device)
            scripted_action[:, :3] = torch.clamp(pos_error * pos_gain, -1.0, 1.0)
            scripted_action[:, 3:6] = torch.clamp(ori_error * ori_gain, -1.0, 1.0)
            scripted_action[:, -1] = -1.0
            
            obs, _, _, _, info = self.env.step(scripted_action)
            f = self._capture_frame()
            if f is not None: self.frames.append(f)

        # --- PHASE 4: RESET CLOCK ---
        if hasattr(env_unwrapped, "_elapsed_steps"):
            env_unwrapped._elapsed_steps.zero_()

        return obs, info

In [224]:

def make_video_env():
    """Single CPU env with render_mode='rgb_array' for video probes."""
    import mani_skill.envs  # noqa: F401
    from hires_vic.envs.maniskill_riemannian import ManiSkillRiemannianWrapper
    use_spd_manifold = True
    use_lie_group = False
    use_llm_prior = False
    use_fixed = True
    env = gym.make(
        'PegInsertionSide-v1',
        num_envs=1,
        obs_mode="state",
        sim_backend="cpu",
        render_mode="rgb_array",
        max_episode_steps=200,
    )
    
    # env = InsertionCurriculumWrapper(env, setup_steps=90)
    env = ManiskillTeleportWrapper(env, setup_steps=50, enable_rendering=True)

    env = ManiSkillRiemannianWrapper(
        env,
        use_spd=use_spd_manifold,
        use_lie_group=use_lie_group,
        use_diag=False,
        use_fixed=use_fixed,
        is_eval=True,
        use_llm_prior=False,
        use_sim2real_obs=True,
        task_metrics_fn=None,
    )
    return env


In [225]:
import os
import torch
import numpy as np
import imageio # Make sure you have imageio installed! (pip install imageio)

try:
    env = make_video_env()
except Exception as e:
    raise RuntimeError('Failed to create ManiSkill3 PegInsertionSide-v1 environment', e)

out_dir = 'outputs'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'peginsertion_primitive.mp4')

print(f"Environment created: {type(env)}")
print("Starting Reset (Running Setup Phases 0-4)...")

# The wrapper handles all 120+ setup steps and frame captures internally!
obs, info = env.reset()

# Access the frames natively from the wrapper
if hasattr(env.env, 'frames') and len(env.env.frames) > 0:
    print(f"Successfully captured {len(env.env.frames)} frames during setup.")
    
    try:
        # Save directly to MP4 using imageio
        imageio.mimsave(out_path, env.env.frames, fps=30)
        print(f"Video safely saved to: {out_path}")
    except Exception as e:
        print(f"Error saving video: {e}")
else:
    print("No frames were captured during reset! Check if enable_rendering=True was passed to the wrapper.")

env.close()

    


True
ManiSkillRiemannianWrapper | SPD=True Diag=False VIC=False Fixed=True LieGroup=False Sim2Real=True | action 8D→17D | obs 29D
Environment created: <class 'hires_vic.envs.maniskill_riemannian.ManiSkillRiemannianWrapper'>
Starting Reset (Running Setup Phases 0-4)...
peg_lengths: tensor([0.1045])
Successfully captured 62 frames during setup.
Video safely saved to: outputs/peginsertion_primitive.mp4


Notes:
- If the scripted policy does not find useful observation keys, inspect `obs` (the observation dict) and adapt `find_key` to the correct keys.
- ManiSkill2 environments often require specialized setup; consult ManiSkill2 docs if env creation fails.
- You can replace `simple_scripted_policy` with your own primitive function that teleports objects or uses lower-level sim APIs.